In [ ]:
import os
import sys
import json
import pickle
import numpy as np

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm, TwoSlopeNorm
import seaborn as sns

fontsize = 9
lw = 0.75
matplotlib.rc('font', **{'family': 'Arial', 'size': fontsize})
matplotlib.rc('axes', **{'linewidth': 0.75, 'labelsize': fontsize})
matplotlib.rc('xtick', **{'labelsize': fontsize})
matplotlib.rc('ytick', **{'labelsize': fontsize})
matplotlib.rc('xtick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.minor', **{'width': lw, 'size':1.5})

In [ ]:
spont_data_file = os.path.join('..','sims','Efield_0.npz')
spont_data = np.load(spont_data_file, allow_pickle=True)

In [ ]:
t,V = spont_data['time'],spont_data['Vsoma']
spike_times = spont_data['spike_times']
ISI = np.diff(spike_times*1e-3)
F = 1 / ISI

In [ ]:
%matplotlib inline
fig,ax = plt.subplots(1, 2, figsize=(7.5,3.5), width_ratios=(1.5,1))
ax[0].plot(t[t<100], V[t<100], 'k', lw=1)
ax[1].plot(spike_times[1:], F, 'k', lw=1)

for a in ax:
    a.set_xlabel('Time (ms)')
ax[1].grid(which='major', axis='y', lw=0.5, ls=':', color=[.6,.6,.6])
ax[0].set_ylabel('Vm (mV)')
ax[1].set_ylabel('Firing rate (1/s)')
sns.despine()
fig.tight_layout()
plt.savefig('purkinje_spontaneous.pdf')

In [ ]:
idx = np.where(spike_times > 800)[0][:2]
t0,t1 = spike_times[idx]
stim_times = np.linspace(t0+0.5, t1-0.5, 20)

In [ ]:
if False:
    config = spont_data['config'].item()
    config['stim']['Efield']['dur'] = 0.1
    for mag in np.r_[60 : 310 : 20]:
        config['stim']['Efield']['mag'] = int(mag)
        for i,stim_time in enumerate(stim_times):
            config['stim']['Efield']['delay'] = stim_time
            outfile = os.path.join('..','configs',f'Efield_{i+1}_{mag}.json')
            json.dump(config, open(outfile,'w'), indent=4)

In [ ]:
stim_data_file = os.path.join('..','sims','Efield_10_200.npz')
if os.path.isfile(stim_data_file):
    stim_data = np.load(stim_data_file, allow_pickle=True)
    kdx = (stim_data['time'] >= t0-5) & (stim_data['time'] <= t1+5)

In [ ]:
jdx = (t >= t0-5) & (t <= t1+5)
fig,ax = plt.subplots(1, 1, figsize=(5,3.5))
cnt = 3
if cnt >= 0:
    ax.plot(t[jdx], V[jdx], 'k', lw=1)
if cnt >= 1:
    ax.vlines(stim_times[9], -80, 50, color=[.2,.8,.2], ls='-', lw=2)
if cnt >= 2:
    try:
        ax.plot(stim_data['time'][kdx], stim_data['Vsoma'][kdx], 'm', lw=1)
    except:
        pass
if cnt >= 3:
    ax.vlines(spike_times[idx], -80, 50, 'r', ls='--', lw=1)
    ax.vlines(stim_times, -80, 50, color=[.6,.6,.6], ls=':', lw=1)
ax.set_ylim([-75, 50])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Vm (mV))')
sns.despine()
fig.tight_layout()
plt.savefig(f'purkinje_Efield_stim_{cnt}.pdf')